# Tecan Infinite M1000 Pro

```{device-card} tecan-infinite-m1000-pro
```

| Property | Value |
| --- | --- |
| Connection | USB |
| Experimental operations | Plate carrier, temperature, 600 nm absorbance |

**Experimental:** tray motion was reported successful after initialization, but an earlier opening attempt produced abnormal stuttering. Stop immediately if movement is abnormal. Temperature control, absorbance acquisition and the updated sequence handling have only been tested offline; they are not hardware-verified.

**Do not operate the tray:** opening has stuttered after a power cycle. Initialization and safe operation are under investigation; a reported position does not prove that the tray moved correctly.

## Connection and safety

Connect the reader over USB using a driver supported by PyLabRobot. On Windows, installing a libusbK driver may prevent i-control from communicating until its Tecan driver is restored. Let the reader complete its normal startup; ensure the carrier path is clear. Obtain appropriate approval before each hardware operation. After an error or unexpected movement, stop, inspect the instrument and resolve its state before reconnecting.

## Setup

`setup()` connects, initializes internal axes and replays the 89-command i-control startup exchange. **Experimental: this revised sequence has not been tested on hardware.** It changes instrument modes and may move internal mechanisms. Check that the instrument is clear, obtain explicit approval for each run, and supervise it. A failed or interrupted setup leaves instrument state uncertain.

In [2]:
from pylabrobot.tecan import InfiniteM1000Pro

reader = InfiniteM1000Pro()
await reader.setup()

2026-11-19 13:20:25,220 - pylabrobot.io.usb - INFO - Finding USB device...
2026-11-19 13:20:25,234 - pylabrobot.io.usb - INFO - Found USB device.
2026-11-19 13:20:25,236 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :  0x200 (512 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :  0x200 (512 bytes)
       bInterval        :    0x0
2026-11-19 13:20:25,237 - pylabrobot.tecan.infinite_m1000_pro - WARNING - Tray opening has produced abnormal stuttering after power-up; do not operate until initializatio

## Initialization compatibility

`setup()` already performs initialization. The following call is idempotent in the same successful session; it is not a recovery procedure. After an error or interruption, stop and assess the instrument before reconnecting.

In [3]:
await reader.initialize()

## Open

Ensure the carrier path is clear before moving it out.

In [4]:
await reader.open()

2026-11-19 13:20:59,178 - pylabrobot.tecan.infinite_m1000_pro - INFO - Moving carrier to OUT
2026-11-19 13:21:02,253 - pylabrobot.tecan.infinite_m1000_pro - INFO - Reader reports carrier OUT


## Close

Place a compatible plate on the carrier if needed; keep the carrier path clear.

In [5]:
await reader.close()

2026-11-19 13:21:02,975 - pylabrobot.tecan.infinite_m1000_pro - INFO - Moving carrier to IN
2026-11-19 13:21:05,778 - pylabrobot.tecan.infinite_m1000_pro - INFO - Reader reports carrier IN


## Reader status

Query the instrument-reported status. This does not establish whether mechanisms have physically stopped.

In [6]:
status = await reader.request_instrument_status()
status

'standby'

## Read absorbance

The current acquisition supports **only** the captured, uncovered physical GRE96ft 96-well plate, 600 nm and all wells. Place that physical plate in the carrier, close it, and supervise the read. The resource below is used only as a logical set of well names; it is not a declaration that a Bio-Rad plate matches the instrument's fixed coordinates. Do not use a different physical plate.

Omit `wells` or pass `None` for the full plate. `wells=[]` and partial selections are rejected. The sequence has not yet been tested with the revised startup/sequence handling. Results are not automatically blank corrected.

In [7]:
from pylabrobot.resources.biorad.plates import biorad_96_wellplate_200uL_Vb

well_map = biorad_96_wellplate_200uL_Vb("well_map_only")
result = await reader.read_absorbance(well_map, wavelength=600, plate_definition="GRE96ft")
result.data

{'A1': -0.0005317230803703499,
 'A2': -0.0010595844819391797,
 'A3': -0.00024471897886400016,
 'A4': -0.00037608277714544237,
 'A5': -0.000544830424707877,
 'A6': 0.0003126210472181649,
 'A7': -3.270052734874497e-05,
 'A8': 0.00017961021178877805,
 'A9': 0.00014628086915289282,
 'A10': 0.0005483353197674244,
 'A11': 0.0003332936594761511,
 'A12': 0.00011294969348789374,
 'B1': -0.0010488969835251943,
 'B2': -0.0014615501386035454,
 'B3': -0.00047484866343506447,
 'B4': -0.0007449907038827176,
 'B5': -0.0006985886498867202,
 'B6': 7.662557053661698e-05,
 'B7': -0.0002959656519710851,
 'B8': 9.461871208239192e-05,
 'B9': -0.00010234314973577141,
 'B10': 0.00040012538656988253,
 'B11': -1.8135274090806392e-05,
 'B12': -0.00018255156978118922,
 'C1': 0.0004052620408180183,
 'C2': 0.0004096451849870039,
 'C3': 0.0002879057506890589,
 'C4': -4.732959038160402e-05,
 'C5': 0.00010589239534156452,
 'C6': 4.2808917058513526e-05,
 'C7': -0.0002661024314267683,
 'C8': 0.00016805048421707244,
 'C9'

## Read fluorescence — experimental reference only

**Hardware-unverified.** This fixed method is based on one i-control capture and simulated transport tests. It moves internal mechanisms. Before running, confirm the physical plate is an **uncovered GRE96ft 96-well plate**, the carrier is IN, and the instrument is clear; obtain per-run approval and supervise the operation. Stop if there is abnormal sound or movement. The `well_map` created above supplies logical well names only; it does not indicate that a Bio-Rad plate is physically compatible.

This example reads all 96 wells with top optics, 230 nm excitation, 280 nm emission, 5 nm bandwidths, gain 100 and 50 flashes. Wavelengths, bandwidths, gain, flashes, timing, Z and well subsets are parameters; any i-control plate definition can be used. Treat the returned values as experimental, not independently validated quantitative measurements.

In [ ]:
fluorescence = await reader.read_fluorescence(
  well_map, None, 230, 280, plate_definition="GRE96ft", gain=100, flashes=50
)
fluorescence.data

## Read temperature

This operation is experimental. Use only with an appropriately supervised, approved session.

In [10]:
temperature = await reader.request_current_temperature()
temperature

30.1

## Enable temperature control

**Not hardware-verified.** A target in the range 20–40 °C is accepted. This changes the reader state; use only with explicit approval and appropriate supervision.

In [9]:
await reader.set_temperature(30.0)

## Disable temperature control

Stop heating or temperature regulation when it is no longer needed.

In [11]:
await reader.stop_temperature_control()

## Disconnect

Disconnecting does not move the carrier.

In [12]:
await reader.stop()

2026-11-19 13:22:04,669 - pylabrobot.io.usb - WARNING - Closing connection to USB device.
2026-11-19 13:22:04,671 - pylabrobot.tecan.infinite_m1000_pro - INFO - Disconnected from Tecan Infinite M1000 Pro
